# 5. Проверка устойчивости спецификаций

**Цель:** вместо несуществующего в публичном API многоагентного интерфейса сопоставить несколько допустимых спецификаций `GraphEVTPipeline`. Таблица явно показывает, какие выводы устойчивы к порогу хвоста, вероятности alarm и persistence.

In [1]:
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
from graph_evt_agent import EVTConfig, GraphConfig, GraphEVTPipeline, InputConfig

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/generated/additive_fractal_series.csv").exists())
DATA = ROOT / "data/generated/additive_fractal_series.csv"
SEED = 42
df = pd.read_csv(DATA)
x = df["value"].to_numpy()
event_mask = df["is_extreme"].to_numpy(dtype=bool)
assert np.isfinite(x).all() and x.ndim == 1
assert event_mask.shape == x.shape and event_mask.any()
event_start = int(np.flatnonzero(event_mask)[0])
event_end = int(np.flatnonzero(event_mask)[-1])

In [2]:
BASELINE_SIZE = event_start
specifications = [
    {"role": "sensitive", "tail_quantile": 0.90, "alarm_probability": 0.995, "persistence": 1},
    {"role": "primary", "tail_quantile": 0.95, "alarm_probability": 0.999, "persistence": 3},
    {"role": "conservative", "tail_quantile": 0.975, "alarm_probability": 0.9995, "persistence": 5},
]
rows = []
for spec in specifications:
    evt = EVTConfig(
        baseline_size=BASELINE_SIZE,
        tail_quantile=spec["tail_quantile"],
        alarm_probability=spec["alarm_probability"],
        persistence=spec["persistence"],
    )
    run = GraphEVTPipeline(
        evt,
        graph=GraphConfig(random_state=SEED),
        input_config=InputConfig(missing="error"),
        verbose=True,
    ).run_detailed(x)
    detection = run.detection
    alarm_index = detection.time_index
    detected_in_labeled_event = alarm_index is not None and bool(event_mask[int(alarm_index)])
    rows.append({
        **spec,
        "alarm_threshold": detection.alarm_threshold,
        "post_baseline_exceedances": int(np.sum(detection.indicator[BASELINE_SIZE:] > detection.alarm_threshold)),
        "labeled_event_exceedances": int(np.sum(detection.indicator[event_mask] > detection.alarm_threshold)),
        "detected": detection.detected,
        "first_persistent_index": alarm_index,
        "detected_in_labeled_event": detected_in_labeled_event,
        "detection_delay_samples": int(alarm_index) - event_start if detected_in_labeled_event else None,
        "graph_available": run.graph is not None,
        "ranking_available": run.ranking is not None,
    })
sensitivity = pd.DataFrame(rows)
sensitivity

Graph-EVT pipeline:   0%|          | 0/3 [00:00<?, ?it/s]

Graph-EVT pipeline:   0%|          | 0/3 [00:00<?, ?it/s]

Graph-EVT pipeline:   0%|          | 0/3 [00:00<?, ?it/s]

,role,tail_quantile,alarm_probability,persistence,alarm_threshold,post_baseline_exceedances,labeled_event_exceedances,detected,first_persistent_index,detected_in_labeled_event,detection_delay_samples,graph_available,ranking_available
0,sensitive,0.900,0.9950,1,1.920919,36,31,True,1088,True,88,False,False
1,primary,0.950,0.9990,3,2.066650,31,29,True,1124,True,124,False,False
2,conservative,0.975,0.9995,5,2.103276,29,28,True,1124,True,124,False,False


In [3]:
artifact_dir = ROOT / "results/1-d"
artifact_dir.mkdir(parents=True, exist_ok=True)
sensitivity.to_csv(artifact_dir / "05_specification_review.csv", index=False)

agreement = {
    "all_detected": bool(sensitivity["detected"].all()),
    "all_detected_in_labeled_event": bool(sensitivity["detected_in_labeled_event"].all()),
    "same_first_index": bool(sensitivity["first_persistent_index"].nunique(dropna=False) == 1),
    "labeled_event_start": event_start,
    "labeled_event_end": event_end,
    "baseline_selection": "labeled_event_start_oracle_assisted",
    "library_version": version("graph-evt-agent"),
}
agreement

{'all_detected': True,
 'all_detected_in_labeled_event': True,
 'same_first_index': False,
 'labeled_event_start': 1000,
 'labeled_event_end': 1149,
 'baseline_selection': 'labeled_event_start_oracle_assisted',
 'library_version': '0.1.0'}

Разные строки — не независимые агенты, а прозрачные статистические спецификации. Во всех трёх вариантах срабатывание попадает в размеченный интервал события, но задержка и число превышений зависят от настроек; совпадение по факту обнаружения не доказывает истинность метода.

Для синтетического примера начало `is_extreme` используется как oracle-assisted граница чистого baseline, а метка — только для оценки срабатывания. На неразмеченных реальных данных такую границу нужно задавать без доступа к будущей метке. Для одномерного ряда библиотека закономерно не строит граф и ranking.

**Самопроверка:** какой вариант быстрее реагирует? Как persistence влияет на задержку и число превышений? Почему выбор baseline по известной метке нельзя переносить напрямую на реальные данные?